# Ejercicios Spark

### Ejercicio 1 - Configuración de SparkSession Personalizada

In [13]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("MiPrimerAnalisis")  # Nombre que aparece en Spark UI
    .config("spark.executor.memory", "1g")  # RAM por executor
    .config("spark.driver.memory", "1g")  # RAM para el driver
    .config("spark.sql.shuffle.partitions", "8")  # Particiones para shuffle
    .getOrCreate())

spark

### Ejercicio 2 Operaciones básicas con los RDDs

In [14]:
numeros = list(range(1, 21))  # [1, 2, 3, ..., 20]
sc = spark.sparkContext
rdd = sc.parallelize(range(1, 21))

conteo = rdd.count()
print(f"La cantidad de números en la lista ´numeros´ es: {conteo}")

F5 = rdd.take(5)
print(f"Los primeros cinco elementos de la lista son: {F5}")

suma = rdd.sum()
print(f"La suma total de los elementos en la lista es: {suma}")

max = rdd.max()
min = rdd.min()
print(f"El valor máximo de la lista es: {max}, mientras que el mínimo es: {min}")

La cantidad de números en la lista ´numeros´ es: 20
Los primeros cinco elementos de la lista son: [1, 2, 3, 4, 5]
La suma total de los elementos en la lista es: 210
El valor máximo de la lista es: 20, mientras que el mínimo es: 1


### Ejercicio 3: Transformaciones map y filter

In [15]:
rdd = sc.parallelize(range(1, 51))

x3 = rdd.map(lambda x: x * 3)
print(f"\nNúmeros después de multiplicar por 3: {x3.collect()}")

may50 = x3.filter(lambda x: x>50)
print(f"\nNúmeros mayores a 50: {may50.collect()}")

contmay50 = may50.count()
print(f"\nTotal de numeros mayores a 50: {contmay50}")

F10 = may50.take(10)
print(f"\nDe la lista anterior, los primeros 10...: {F10}")



Números después de multiplicar por 3: [3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45, 48, 51, 54, 57, 60, 63, 66, 69, 72, 75, 78, 81, 84, 87, 90, 93, 96, 99, 102, 105, 108, 111, 114, 117, 120, 123, 126, 129, 132, 135, 138, 141, 144, 147, 150]

Números mayores a 50: [51, 54, 57, 60, 63, 66, 69, 72, 75, 78, 81, 84, 87, 90, 93, 96, 99, 102, 105, 108, 111, 114, 117, 120, 123, 126, 129, 132, 135, 138, 141, 144, 147, 150]

Total de numeros mayores a 50: 34

De la lista anterior, los primeros 10...: [51, 54, 57, 60, 63, 66, 69, 72, 75, 78]


### Ejercicio 4: trabajar con pares clave-valor

In [16]:
rdd = sc.parallelize([
    ("laptop", 1200),
    ("mouse", 25),
    ("laptop", 1100),
    ("teclado", 75),
    ("mouse", 30),
    ("laptop", 1300)
])

vpp = rdd.reduceByKey(lambda a,b: a+b)
print(f"el total en ventas por producto fue el siguiente:")
print(vpp.collect())

cvp = (rdd.map(lambda x: (x[0], 1)).reduceByKey(lambda a,b: a+b))
print(f"El total de ventas de cada producto es el siguiente: \n{cvp.collect()}")

vp = (rdd.mapValues(lambda x: 1).reduceByKey(lambda a,b: a+b))
mvp = vp.reduce(lambda a, b: a if a[1] > b[1] else b)
print(f"El producto con más ventas fue: {mvp}")

el total en ventas por producto fue el siguiente:
[('laptop', 3600), ('mouse', 55), ('teclado', 75)]
El total de ventas de cada producto es el siguiente: 
[('laptop', 3), ('mouse', 2), ('teclado', 1)]
El producto con más ventas fue: ('laptop', 3)


### Ejercicio 5: Lectura y procesamiento de archivo CSV

In [17]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("AnalisisVentas").getOrCreate()
sc = spark.sparkContext

# Construir ruta absoluta
path_relativo = "./DATOS/persons.csv"
path_absoluto = os.path.abspath(path_relativo)
path_final = "file://" + path_absoluto

print(f"Leyendo desde: {path_final}")

# Leer archivo (cada línea es un string)
rdd = sc.textFile(path_final)

Leyendo desde: file:///home/jovyan/DATOS/persons.csv


In [28]:
#Eliminación de encabezado:
header = rdd.first()
print(f"Encabezado: {header}")

# Filtrar todas las líneas que NO sean el encabezado
rdd_no_enc = rdd.filter(lambda linea: linea != header)

rdd_no_enc.collect()

Encabezado: name,age,gender,salary


['John Doe,30,Male,50000',
 'Jane Smith,25,Female,45000',
 'David Johnson,35,Male,60000',
 'Emily Davis,28,Female,52000',
 'Michael Wilson,40,Male,75000',
 'Sarah Brown,32,Female,58000',
 'Robert Lee,29,Male,51000',
 'Lisa Garcia,27,Female,49000',
 'James Martinez,38,Male,70000',
 'Jennifer Rodriguez,26,Female,47000',
 'William Anderson,33,Male,62000',
 'Karen Hernandez,31,Female,55000',
 'Christopher Taylor,37,Male,69000',
 'Mary Gonzalez,24,Female,44000',
 'Matthew Davis,36,Male,67000',
 'Patricia White,29,Female,50000',
 'Daniel Miller,34,Male,64000',
 'Elizabeth Jackson,30,Female,52000',
 'Joseph Harris,28,Male,53000',
 'Linda Martin,39,Female,71000']

In [31]:
#Extracción de nombres y edades:
rdd_nom_eda = rdd_no_enc.map(lambda linea: (
    linea.split(",")[0].strip(),
    linea.split(",")[1] 
))

rdd_nom_eda.collect()

[('John Doe', '30'),
 ('Jane Smith', '25'),
 ('David Johnson', '35'),
 ('Emily Davis', '28'),
 ('Michael Wilson', '40'),
 ('Sarah Brown', '32'),
 ('Robert Lee', '29'),
 ('Lisa Garcia', '27'),
 ('James Martinez', '38'),
 ('Jennifer Rodriguez', '26'),
 ('William Anderson', '33'),
 ('Karen Hernandez', '31'),
 ('Christopher Taylor', '37'),
 ('Mary Gonzalez', '24'),
 ('Matthew Davis', '36'),
 ('Patricia White', '29'),
 ('Daniel Miller', '34'),
 ('Elizabeth Jackson', '30'),
 ('Joseph Harris', '28'),
 ('Linda Martin', '39')]

In [37]:
#Personas con más de 30 años
may30 = rdd_nom_eda.filter(
    lambda persona: int(persona[1]) > 30
)

may30.collect()

[('David Johnson', '35'),
 ('Michael Wilson', '40'),
 ('Sarah Brown', '32'),
 ('James Martinez', '38'),
 ('William Anderson', '33'),
 ('Karen Hernandez', '31'),
 ('Christopher Taylor', '37'),
 ('Matthew Davis', '36'),
 ('Daniel Miller', '34'),
 ('Linda Martin', '39')]

In [36]:
#Cuántos cumplen la condición
print(f"En la lista filtrada anterior, hay: {may30.count()} registros")

En la lista filtrada anterior, hay: 10 registros


### Ejercicio 6: Optimización - Evitar múltiples split()

###### Código ineficiente

~~~ Python
# ❌ INEFICIENTE: split() se ejecuta 3 veces por línea
nombres = rdd.map(lambda x: x.split(",")[0])
edades = rdd.map(lambda x: x.split(",")[1])
ciudades = rdd.map(lambda x: x.split(",")[2])
~~~

Problema: el código ejecuta tres veces split por línea, aunque esto no representa un problema de primer momento con el dataframe actual con sus dimensiones tan reducidas, sí puede ralentizar gravemente el procesamiento de datasets de mayor magnitud.

Nótese que en el dtaaframe actual no existe una columna de ciudades, en su lugar se va a utilizar la columna "salario". 

In [40]:
def parsear_linea(linea):
    columnas = linea.split(",")
    nombre = columnas[0].strip()
    edad = columnas[1]
    salario = columnas[3]
    return (nombre, edad, salario)

rdd_opt = rdd_no_enc.map(parsear_linea)
rdd_opt.collect()

[('John Doe', '30', '50000'),
 ('Jane Smith', '25', '45000'),
 ('David Johnson', '35', '60000'),
 ('Emily Davis', '28', '52000'),
 ('Michael Wilson', '40', '75000'),
 ('Sarah Brown', '32', '58000'),
 ('Robert Lee', '29', '51000'),
 ('Lisa Garcia', '27', '49000'),
 ('James Martinez', '38', '70000'),
 ('Jennifer Rodriguez', '26', '47000'),
 ('William Anderson', '33', '62000'),
 ('Karen Hernandez', '31', '55000'),
 ('Christopher Taylor', '37', '69000'),
 ('Mary Gonzalez', '24', '44000'),
 ('Matthew Davis', '36', '67000'),
 ('Patricia White', '29', '50000'),
 ('Daniel Miller', '34', '64000'),
 ('Elizabeth Jackson', '30', '52000'),
 ('Joseph Harris', '28', '53000'),
 ('Linda Martin', '39', '71000')]

La solución es, en términos de procesamiento, más eficiente en el sentido de que ejecuta un split una sola vez por registro, identificando el separador `(,)` y tomándolo de referencia para identificar cada dato individual.

### Ejercicio 7: Debugging - Encuentra y corrige errores.

Código con errores: 

~~~ python
from pyspark.sql import SparkSession

# Error 1: Falta configuración
spark = SparkSession.builder.getOrCreate()

# Error 2: Ruta incorrecta para archivo local
rdd = sc.textFile("./datos.csv")

# Error 3: Uso de collect() en dataset grande
datos = rdd.collect()
for linea in datos:
    print(linea)

# Error 4: No se cierra la sesión
~~~

Errores encontrados: 
- El perfil no tiene configuración de `SparkContext`, lo que puede impedir la ejecución de operaciones de bajo nivel. 
- La ruta resulta ambigua para spark, ya que este intenta identificar archivos en HDFS, por lo que al buscar en `./datos.csv` no encontrará ningún archivo compatible al no encontrar el directorio. Es necesario indicar una ruta absoluta para indicar que el archivo está en el sistema local.
- Collect se encarga de devolver los datos enteros de un dataframe en forma de variable, lo que, dependiendo de la dimensión de la variable, puede resultar muy pesado de procesar; usar `take(#)` con un valor mesurado sería más digerible para Spark
- La sesión permanece abierta a falta de un comando para terminarla; si existe una falla en el perfil, configurarlo puede ser mucho más difícil si no se cierra por principio. 

In [42]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("correccion").getOrCreate()
#corrección 1:
sc = spark.sparkContext

#corrección 2 (dataframe trabajado, de referencia)
path_relativo = "./DATOS/persons.csv"
path_absoluto = os.path.abspath(path_relativo)
path_final = "file://" + path_absoluto
rdd = sc.textFile(path_final) #lectura del dataframe

#corrección 3: 
muestra = rdd.take(5)
muestra

['name,age,gender,salary',
 'John Doe,30,Male,50000',
 'Jane Smith,25,Female,45000',
 'David Johnson,35,Male,60000',
 'Emily Davis,28,Female,52000']

In [45]:
#corrección 4:
spark.stop()